# RoboBarista – sichere Anfahrwege zu den sieben Stationen

Hier simulieren wir auf der **Artly-inspirierten Kaffeeinsel** den virtuellen FR5 bei einer freigegebenen **Vorposition** (*pre-dock*) zu einer Kaffeemaschine.

**Wichtig:** Ein sicherer Anfahrweg bedeutet in diesem Test: Das FR5-Modell berührt entlang aller zusätzlich geprüften Gelenkzwischenstellungen keine Geräte und die vereinfachten Hüllkörper seines getragenen Werkzeugs bleiben außerhalb der Maschinen. Eine **endgültige Andock- oder Pressbewegung ist NICHT freigegeben**. Sind keine Anfahrwege nachweisbar, bleiben die Stationen bewusst gesperrt.

Für jede der sieben Werkzeug-/Geräteschnittstellen werden verschiedene Vorpositionen, Armhaltungen und ein kompletter Hin-/Rückweg zur Arbeitsmitte geprüft. Nur die bestandenen Wege erscheinen als Bewegung im Film.

**Laufzeit → Alle ausführen.** Keine GPU nötig.


In [ ]:
# 1. Repo laden
from pathlib import Path
import subprocess,os,sys
repo=Path('/content/robobarista-vision')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1',
      'https://github.com/Coffee-Tobi/robocasa.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
os.chdir(repo)
print(repo)


In [ ]:
# 2. Dependencies
subprocess.run(['apt-get','-qq','update'],check=True)
subprocess.run(['apt-get','-qq','install','-y','libosmesa6-dev',
                'libgl1','libglx-mesa0','libgl1-mesa-dri','ffmpeg'],check=True)
subprocess.run([sys.executable,'-m','pip','-q','install','-r','requirements.txt'],check=True)


In [ ]:
# 3. Physikalische Roboter-Kollisionsformen und gesamte Werkzeug-Huelle
#    ueber gepruefte Gelenkzwischenstellungen laufen lassen.
from pathlib import Path
env=os.environ.copy()
env['MUJOCO_GL']='egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
if env['MUJOCO_GL']=='osmesa': env['PYOPENGL_PLATFORM']='osmesa'
args=[sys.executable,'-u','-m','robobarista.safe_station_staging_demo',
      '--seed','14','--frames-per-station','12',
      '--width','960','--height','720','--output','runs/safe_staging']
with subprocess.Popen(args,cwd=str(repo),env=env,stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT,text=True,errors='replace') as p:
    for line in p.stdout:
        print(line,end='',flush=True)
    exit_code=p.wait()
print('Exit-Code:',exit_code)
if exit_code:
    print('Gegebenenfalls wurden noch keine freigegebenen Positionen gefunden.')


In [ ]:
# 4. Sichere FR5-Bewegungen und explizit GESPERRTE Stationen anzeigen
from IPython.display import display,Video,Image
import json,pandas as pd
out=Path(repo)/'runs/safe_staging'
mp4=out/'safe_staging.mp4'
gif=out/'safe_staging.gif'
if mp4.exists():
    try:display(Video(str(mp4),embed=True,width=930))
    except (TypeError,OSError,ValueError):
        if gif.exists():display(Image(filename=str(gif),format='gif'))
elif gif.exists():
    display(Image(filename=str(gif),format='gif'))
r=out/'safe_staging_report.json'
if r.exists():
    report=json.loads(r.read_text())
    print('Freigegebene Vorpositionen:',report['pre_dock_allowed_count'],'/',7)
    print('Noch GESPERRT:',report['blocked_count'])
    print('Mechanisches Andocken freigegeben:',report['actual_contact_docking_approved'])
    display(pd.DataFrame([
        {'Gerät/Ziel':name,
         'Vorposition':v['status'],
         'IK-Versuche mit Pose':v['attempts']['ik_reached'],
         'Kontaktfreie IK-Kandidaten':v['attempts']['pose_contact_free'],
         'Vollständig geprüfte Hin-/Rueckwege':v['attempts']['safe_full_joint_routes']}
        for name,v in report['stage_poses'].items()
    ]))


In [ ]:
# 5. Video und ausführlichen Bericht sichern
from IPython.display import FileLink,display
import zipfile
path=out/'RoboBarista_Stationsanfahrten.zip'
with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ('safe_staging.mp4','safe_staging.gif','safe_staging_report.json'):
        f=out/name
        if f.exists():z.write(f,name)
display(FileLink(str(path)))


## Danach

Dies ist der **erste konservative Vorpositions-/Kontaktfilter**. Auch bestätigte Wege sind **nur kinematisch** und werden vor realer Hardware durch exaktes Geräte-/Greifer-CAD, echten Werkzeug-Anbau, motorische Geschwindigkeit/Beschleunigung, sensorbasierte Stopplogik und mechanische Dockingprüfung ergänzt.

Die eigenständige **[Kamera-zu-FR5-Rekonstruktion](https://colab.research.google.com/github/Coffee-Tobi/robocasa/blob/main/COLAB_SAFE_REPLAY.ipynb)** zeigt parallel, wie ein Apriltag-basiert beobachteter zufälliger Weg durch den FR5 nachgestellt wird. Beide Experimente bleiben bewusst getrennt, um ihre jeweiligen Fehlerursachen erkennen zu können.
